# 61 - First 10s Strict Pipeline Gate

This notebook checks that the cleaned package surface still supports the current executable path. It runs both the fixed-R strict runner and the adaptive-R wrapper on the first 10 seconds of `UltraTimTrack_test.mp4`, then validates the output arrays.

In [1]:
from __future__ import annotations

import csv
import json
import math
import subprocess
import sys
from pathlib import Path

import cv2
import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / 'ultrasound_tracker').exists():
    ROOT = next(p for p in Path.cwd().resolve().parents if (p / 'ultrasound_tracker').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

VIDEO = ROOT / 'data' / 'raw' / 'UltraTimTrack_test.mp4'
ROI_PATH = ROOT / 'data' / 'rois' / 'UltraTimTrack_test_rois.json'
UTT_EXPORT = Path('/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat')
OUT_ROOT = ROOT / 'results' / 'notebook61_first10s_strict_pipeline_gate'

assert VIDEO.exists(), VIDEO
assert ROI_PATH.exists(), ROI_PATH
assert UTT_EXPORT.exists(), UTT_EXPORT

cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened(), VIDEO
fps = float(cap.get(cv2.CAP_PROP_FPS))
n_video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

LIMIT_FRAMES = int(round(fps * 10.0))
LIMIT_FRAMES = min(LIMIT_FRAMES, n_video_frames)

print(json.dumps({
    'video': str(VIDEO),
    'fps': fps,
    'n_video_frames': n_video_frames,
    'limit_frames': LIMIT_FRAMES,
    'duration_s': (LIMIT_FRAMES - 1) / fps,
}, indent=2))

{
  "video": "/Users/grosbedou/PycharmProjects/NDORMS/data/raw/UltraTimTrack_test.mp4",
  "fps": 33.341,
  "n_video_frames": 2667,
  "limit_frames": 333,
  "duration_s": 9.957709726762845
}


In [2]:
import ultrasound_tracker as ut

print('ultrasound_tracker', ut.__version__)
print('public names:', ', '.join(ut.__all__))

for forbidden in ['FascicleKalman', 'KLTTracker', 'HoughDetector', 'FrangiDetector', 'SpeckleTracker', 'DenseFlowTracker']:
    assert forbidden not in ut.__all__, forbidden
    assert not hasattr(ut, forbidden), forbidden

required = ['run_timtrack_geofeatures_from_video', 'run_one_step_affine_video', 'run_matlab_aponeurosis_state_video', 'run_matlab_2state_kalman', 'SpeckleConfidenceConfig']
for name in required:
    assert hasattr(ut, name), name

ultrasound_tracker 2.0.0
public names: FascicleSeedScoringConfig, MatlabTwoStateKalmanConfig, SpeckleConfidenceConfig, UltraTrackKLTConfig, cluster_seed_candidates, combine_confidence_metrics, compute_feature_detection_reliability, compute_geometry_stability, compute_motion_consistency, compute_speckle_coherence, confidence_to_r_scale, detect_timtrack_geofeature_from_image, extract_fascicle_seed_candidates, fascicle_segment_from_geofeature, geometry, make_matlab_apox, propagate_cumulative_affines, read_gray_frames, roi, run_matlab_2state_kalman, run_matlab_aponeurosis_state_video, run_one_step_affine_video, run_timtrack_geofeatures_from_video, select_autonomous_fascicle_seed


In [3]:
def run_command(label: str, cmd: list[str], timeout_s: int = 900) -> None:
    print('\n' + '=' * 80)
    print(label)
    print(' '.join(str(x) for x in cmd))
    proc = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, timeout=timeout_s)
    print(proc.stdout[-6000:])
    if proc.stderr.strip():
        print('STDERR tail:')
        print(proc.stderr[-4000:])
    if proc.returncode != 0:
        raise RuntimeError(f'{label} failed with return code {proc.returncode}')

strict_cmd = [
    sys.executable,
    str(ROOT / 'scripts' / 'run_strict_ultratimtrack_video.py'),
    str(VIDEO),
    '--roi-path', str(ROI_PATH),
    '--utt-export', str(UTT_EXPORT),
    '--limit', str(LIMIT_FRAMES),
    '--seed-frames', '11',
    '--results-dir', str(OUT_ROOT / 'fixed_r'),
    '--no-annotated-video',
    '--save-overlays', '1',
    '--no-time-series-plot',
    '--progress-every', '100',
]

adaptive_cmd = [
    sys.executable,
    str(ROOT / 'scripts' / 'run_ultratimtrack_adaptive_confidence.py'),
    str(VIDEO),
    '--roi-path', str(ROI_PATH),
    '--utt-export', str(UTT_EXPORT),
    '--limit', str(LIMIT_FRAMES),
    '--seed-frames', '11',
    '--results-dir', str(OUT_ROOT / 'adaptive_r'),
    '--adaptive-r',
    '--save-confidence-plots',
    '--no-annotated-video',
    '--save-overlays', '1',
    '--no-time-series-plot',
    '--progress-every', '100',
]

run_command('fixed-R strict runner', strict_cmd)
run_command('adaptive-R wrapper', adaptive_cmd)


fixed-R strict runner
/Users/grosbedou/PycharmProjects/NDORMS/.venv/bin/python /Users/grosbedou/PycharmProjects/NDORMS/scripts/run_strict_ultratimtrack_video.py /Users/grosbedou/PycharmProjects/NDORMS/data/raw/UltraTimTrack_test.mp4 --roi-path /Users/grosbedou/PycharmProjects/NDORMS/data/rois/UltraTimTrack_test_rois.json --utt-export /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat --limit 333 --seed-frames 11 --results-dir /Users/grosbedou/PycharmProjects/NDORMS/results/notebook61_first10s_strict_pipeline_gate/fixed_r --no-annotated-video --save-overlays 1 --no-time-series-plot --progress-every 100


Loading ROIs: /Users/grosbedou/PycharmProjects/NDORMS/data/rois/UltraTimTrack_test_rois.json

Running TimTrack image stream...
TimTrack image geofeatures processed 100
TimTrack image geofeatures processed 200
TimTrack image geofeatures processed 300

Selecting autonomous fascicle seed...
Selected seed alpha: 17.500 deg (a17.50_x350_l75)

Estimating fascicle KLT affines...
one-step KLT processed 101/333
one-step KLT processed 201/333
one-step KLT processed 301/333
one-step KLT processed 333/333

Running aponeurosis state estimator...
aponeurosis state processed 101/333
aponeurosis state processed 201/333
aponeurosis state processed 301/333
aponeurosis state processed 333/333

Running 2-state fascicle Kalman...

Done.
CSV: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook61_first10s_strict_pipeline_gate/fixed_r/UltraTimTrack_test/UltraTimTrack_test_strict_FL_PEN_ANG.csv
NPZ: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook61_first10s_strict_pipeline_gate/fixed_r/UltraTimTr

Loading ROIs: /Users/grosbedou/PycharmProjects/NDORMS/data/rois/UltraTimTrack_test_rois.json

Running TimTrack image stream...
TimTrack image geofeatures processed 100
TimTrack image geofeatures processed 200
TimTrack image geofeatures processed 300

Selecting autonomous fascicle seed...
Selected seed alpha: 17.500 deg (a17.50_x350_l75)

Estimating fascicle KLT affines...
one-step KLT processed 101/333
one-step KLT processed 201/333
one-step KLT processed 301/333
one-step KLT processed 333/333

Running aponeurosis state estimator...
aponeurosis state processed 101/333
aponeurosis state processed 201/333
aponeurosis state processed 301/333
aponeurosis state processed 333/333

Computing ultrasound confidence metrics...

Running 2-state fascicle Kalman...

Done.
CSV: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook61_first10s_strict_pipeline_gate/adaptive_r/UltraTimTrack_test/UltraTimTrack_test_strict_FL_PEN_ANG.csv
NPZ: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook61_f

In [4]:
def load_run(root: Path, stem: str) -> dict[str, np.ndarray]:
    run_dir = root / stem
    npz_path = run_dir / f'{stem}_strict_results.npz'
    csv_path = run_dir / f'{stem}_strict_FL_PEN_ANG.csv'
    metadata_path = run_dir / f'{stem}_strict_metadata.json'
    assert npz_path.exists(), npz_path
    assert csv_path.exists(), csv_path
    assert metadata_path.exists(), metadata_path
    with np.load(npz_path) as data:
        arrays = {key: data[key] for key in data.files}
    with csv_path.open(newline='') as f:
        rows = list(csv.DictReader(f))
    arrays['_csv_row_count'] = np.asarray(len(rows))
    arrays['_metadata'] = np.asarray(json.loads(metadata_path.read_text()))
    arrays['_npz_path'] = np.asarray(str(npz_path))
    return arrays

fixed = load_run(OUT_ROOT / 'fixed_r', VIDEO.stem)
adaptive = load_run(OUT_ROOT / 'adaptive_r', VIDEO.stem)

print('fixed npz:', fixed['_npz_path'].item())
print('adaptive npz:', adaptive['_npz_path'].item())

fixed npz: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook61_first10s_strict_pipeline_gate/fixed_r/UltraTimTrack_test/UltraTimTrack_test_strict_results.npz
adaptive npz: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook61_first10s_strict_pipeline_gate/adaptive_r/UltraTimTrack_test/UltraTimTrack_test_strict_results.npz


In [5]:
def finite_fraction(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    return float(np.isfinite(values).mean())

def summarize_run(label: str, arrays: dict[str, np.ndarray]) -> dict[str, float]:
    n = len(arrays['frame'])
    assert n == LIMIT_FRAMES, (label, n, LIMIT_FRAMES)
    assert int(arrays['_csv_row_count']) == LIMIT_FRAMES, label
    assert np.all(np.diff(arrays['frame']) == 1), label
    assert np.all(np.diff(arrays['time_s']) > 0), label

    summary = {
        'n_frames': float(n),
        'duration_s': float(arrays['time_s'][-1]),
        'ANG_median_deg': float(np.nanmedian(arrays['ANG_deg'])),
        'ANG_range_deg': float(np.nanmax(arrays['ANG_deg']) - np.nanmin(arrays['ANG_deg'])),
        'PEN_median_deg': float(np.nanmedian(arrays['PEN_deg'])),
        'PEN_range_deg': float(np.nanmax(arrays['PEN_deg']) - np.nanmin(arrays['PEN_deg'])),
        'FL_median_mm': float(np.nanmedian(arrays.get('FL_mm', arrays['FL_px']))),
        'FL_range_mm_or_px': float(np.nanmax(arrays.get('FL_mm', arrays['FL_px'])) - np.nanmin(arrays.get('FL_mm', arrays['FL_px']))),
        'ANG_finite_fraction': finite_fraction(arrays['ANG_deg']),
        'PEN_finite_fraction': finite_fraction(arrays['PEN_deg']),
        'FL_finite_fraction': finite_fraction(arrays.get('FL_mm', arrays['FL_px'])),
    }
    assert summary['ANG_finite_fraction'] > 0.99, label
    assert summary['PEN_finite_fraction'] > 0.99, label
    assert summary['FL_finite_fraction'] > 0.99, label
    assert 5.0 <= summary['ANG_median_deg'] <= 45.0, summary
    assert 0.0 <= summary['PEN_median_deg'] <= 45.0, summary
    assert summary['FL_median_mm'] > 0.0, summary
    return summary

summaries = {
    'fixed': summarize_run('fixed', fixed),
    'adaptive': summarize_run('adaptive', adaptive),
}

print(json.dumps(summaries, indent=2))

{
  "fixed": {
    "n_frames": 333.0,
    "duration_s": 9.957709726762845,
    "ANG_median_deg": 22.641137199374867,
    "ANG_range_deg": 11.370201592530663,
    "PEN_median_deg": 23.621919277974655,
    "PEN_range_deg": 10.482564421001712,
    "FL_median_mm": 59.982469865653776,
    "FL_range_mm_or_px": 28.502027201053075,
    "ANG_finite_fraction": 1.0,
    "PEN_finite_fraction": 1.0,
    "FL_finite_fraction": 1.0
  },
  "adaptive": {
    "n_frames": 333.0,
    "duration_s": 9.957709726762845,
    "ANG_median_deg": 22.837804189703718,
    "ANG_range_deg": 11.744109315916148,
    "PEN_median_deg": 23.824998191423646,
    "PEN_range_deg": 10.856461618382411,
    "FL_median_mm": 59.489047138412076,
    "FL_range_mm_or_px": 29.04093475902097,
    "ANG_finite_fraction": 1.0,
    "PEN_finite_fraction": 1.0,
    "FL_finite_fraction": 1.0
  }
}


In [6]:
def rmse(a: np.ndarray, b: np.ndarray) -> float:
    diff = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    return float(np.sqrt(np.nanmean(diff ** 2)))

comparison = {
    'ANG_adaptive_minus_fixed_rmse_deg': rmse(adaptive['ANG_deg'], fixed['ANG_deg']),
    'ANG_adaptive_minus_fixed_max_abs_deg': float(np.nanmax(np.abs(adaptive['ANG_deg'] - fixed['ANG_deg']))),
    'PEN_adaptive_minus_fixed_rmse_deg': rmse(adaptive['PEN_deg'], fixed['PEN_deg']),
    'PEN_adaptive_minus_fixed_max_abs_deg': float(np.nanmax(np.abs(adaptive['PEN_deg'] - fixed['PEN_deg']))),
    'FL_adaptive_minus_fixed_rmse_mm_or_px': rmse(adaptive.get('FL_mm', adaptive['FL_px']), fixed.get('FL_mm', fixed['FL_px'])),
    'FL_adaptive_minus_fixed_max_abs_mm_or_px': float(np.nanmax(np.abs(adaptive.get('FL_mm', adaptive['FL_px']) - fixed.get('FL_mm', fixed['FL_px'])))),
}

if 'combined_confidence' in adaptive:
    conf = adaptive['combined_confidence']
    r_scale = adaptive['r_scale']
    comparison.update({
        'adaptive_confidence_median': float(np.nanmedian(conf)),
        'adaptive_confidence_min': float(np.nanmin(conf)),
        'adaptive_r_scale_median': float(np.nanmedian(r_scale)),
        'adaptive_r_scale_max': float(np.nanmax(r_scale)),
    })
    assert finite_fraction(conf) > 0.99
    assert finite_fraction(r_scale) > 0.99
    assert np.nanmin(conf) >= 0.0
    assert np.nanmax(conf) <= 1.0
    assert np.nanmin(r_scale) > 0.0

assert comparison['ANG_adaptive_minus_fixed_max_abs_deg'] < 5.0, comparison
assert comparison['PEN_adaptive_minus_fixed_max_abs_deg'] < 5.0, comparison

print(json.dumps(comparison, indent=2))

{
  "ANG_adaptive_minus_fixed_rmse_deg": 0.24897314977701776,
  "ANG_adaptive_minus_fixed_max_abs_deg": 0.3739436547234334,
  "PEN_adaptive_minus_fixed_rmse_deg": 0.24897314977701776,
  "PEN_adaptive_minus_fixed_max_abs_deg": 0.3739436547234334,
  "FL_adaptive_minus_fixed_rmse_mm_or_px": 0.4934683283689636,
  "FL_adaptive_minus_fixed_max_abs_mm_or_px": 0.5935340784643586,
  "adaptive_confidence_median": 0.9653835165927557,
  "adaptive_confidence_min": 0.7012834152046284,
  "adaptive_r_scale_median": 0.6255911484706758,
  "adaptive_r_scale_max": 3.683637520685321
}


## Gate conclusion

If all cells above execute, the cleaned package import surface, fixed-R strict runner, adaptive-R wrapper, output files, and first-10-second ANG/PEN/FL arrays are functioning together.